# Chapter 1 — Knowledge Graph Indexing

## 학습 목표
1. **3-layer LPG** 구조 (Source → Chunk → Entity)를 RDF triple과 비교하며 정당화한다.
2. **Ontology-aware extraction 프롬프트**를 설계하고 schema 강제 유무에 따른 4-provider 응답 차이를 측정한다.
3. **Community Detection (Louvain)** 결과를 노드 property로 write-back 한다.

## 전제 조건
- `chapter-00-setup.ipynb` 통과
- Neo4j/DozerDB 인스턴스 + `apoc.*` / `n10s.*` / `gds.*` 권한
- FIBO TTL 발췌: `examples/datasets/fibo_be_minimal.ttl`
- FinDER 데이터셋 (Chapter 0에서 캐싱됨)

## 이번 챕터 데이터 플로우
```
FinDER (test split)  ─┐
                      ├→ IndexingPipeline ─→ (:Source)-[:HAS_CHUNK]->(:Chunk)-[:MENTIONS]->(:Entity)
FIBO TTL ─→ Ontology ─┘                                                    │
                                                                           ▼
                                                                   GDS Louvain ─→ community_id
```

## 0. Setup

Opik 트레이싱 ON. 모든 후속 셀은 트레이스에 기록되어 `teaching-ch01-{user}` 프로젝트에서 확인 가능.

In [ ]:
from pathlib import Path
import os, sys, json
from dotenv import load_dotenv

HERE = Path.cwd()
if HERE.name != 'teaching-resource' and (HERE / 'teaching-resource').exists():
    os.chdir(HERE / 'teaching-resource')
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))
for cand in [Path.cwd() / '.env', Path.cwd().parent / '.env']:
    if cand.exists():
        load_dotenv(cand, override=False); break

from _shared.opik_setup import setup_opik, opik_console_link, teardown_opik
from _shared.providers import compare_providers, available_providers, chat, PROVIDERS
from _shared.finder_loader import by_category, sample_random, sample_per_category

trace_info = setup_opik('01', only_jsonl=not os.getenv('OPIK_API_KEY'))
print('Opik project:', trace_info['project'])
print('Backends    :', trace_info['backends'])
if trace_info['project']:
    print('Opik UI     :', opik_console_link(trace_info['project']))

## 1.1 Source / Chunk / Entity — 3-layer LPG

왜 단일 `(:Entity)` 노드가 아니라 세 계층으로 분리하는가?

| 분리하지 않으면 | 분리하면 |
|---|---|
| 출처 추적 불가 (어느 문서 어느 청크에서 왔는지) | `(:Source)` 노드 한 곳에 메타데이터 응집 |
| 청크 단위 임베딩 재계산 불가 | `(:Chunk)` 단위로 부분 재인덱싱 가능 |
| 동일 엔터티의 중복 노드 폭증 | `(:Entity)`는 dedup, MENTIONS edge로 다대다 |
| Source 삭제 시 cascade 정책 모호 | DETACH DELETE로 명확한 lineage |

**RDF triple과의 비교** — 같은 "문서 D의 chunk C에서 추출된 엔터티 E"를 RDF로 표현하려면 reification(별도 statement 노드)이 필요. LPG는 그냥 edge property로 끝.

### FinDER 코퍼스 한 건 + FIBO 온톨로지 준비

Risk 카테고리에서 1개 record를 골라 인덱싱한다.

In [ ]:
risk = by_category('Risk')
doc = risk[0]
print('record id   :', doc.get('id'))
print('category    :', doc.get('category'))
print('text len    :', len(doc.get('text', '')))
print('question    :', doc.get('question', '')[:200])
print('text head   :', (doc.get('text') or '')[:500])

In [ ]:
from seocho.ontology import Ontology
from _shared.compat import load_ontology

from pathlib import Path as _P
_TTL_CANDIDATES = [
    _P('datasets') / 'fibo_be_minimal.ttl',
    _P('teaching-resource') / 'datasets' / 'fibo_be_minimal.ttl',
    _P('..') / 'examples' / 'datasets' / 'fibo_be_minimal.ttl',
    _P('examples') / 'datasets' / 'fibo_be_minimal.ttl',
]
FIBO_TTL = next((p for p in _TTL_CANDIDATES if p.exists()), _TTL_CANDIDATES[0])
print('using FIBO TTL:', FIBO_TTL)

ontology = load_ontology(str(FIBO_TTL))
# Ontology는 nodes/relationships 를 dict 로 보관
print('classes      :', list(ontology.nodes.keys())[:10])
print('relationships:', list(ontology.relationships.keys())[:10])


### Seocho 클라이언트 + add() — 3-layer 자동 생성

In [ ]:
import os
from pathlib import Path
from seocho import Seocho, create_llm_backend
from seocho.store.graph import Neo4jGraphStore

NEO4J_URI = os.getenv('NEO4J_URI', 'bolt://localhost:7687')
NEO4J_USER = os.getenv('NEO4J_USER', 'neo4j')
NEO4J_PWD  = os.getenv('NEO4J_PASSWORD')
USE_NEO4J  = bool(NEO4J_PWD)

# LLM backend
llm = create_llm_backend(provider='openai', model='gpt-4o-mini')

# Graph store — Neo4j or Ladybug embedded
if USE_NEO4J:
    graph_store = Neo4jGraphStore(uri=NEO4J_URI, user=NEO4J_USER, password=NEO4J_PWD)
else:
    Path('./runs').mkdir(exist_ok=True)
    from seocho.store.graph import LadybugGraphStore
    graph_store = LadybugGraphStore(path='./runs/chapter01.lbug')

client = Seocho(
    ontology=ontology,
    graph_store=graph_store,
    llm=llm,
    workspace_id=f"teaching-ch01-{os.getenv('OPIK_USER', 'anonymous')}",
)
print('client mode:', 'neo4j' if USE_NEO4J else 'ladybug')
print('database   :', client.default_database)


In [ ]:
# 인덱싱 — text 청크가 자동으로 (:Source)-[:HAS_CHUNK]->(:Chunk)-[:MENTIONS]->(:Entity) 로 생성됨
memory = client.add(
    doc['text'],
    metadata={'finder_id': doc.get('id'), 'category': doc.get('category')},
)
print('entities :', len(memory.entities))
print('relations:', len(memory.relationships))
for e in memory.entities[:6]:
    print(f"  - {e.label}: {e.properties.get('name')}")

### 3-layer 카운트 검증 (Cypher 직접 실행)

기대값: Source=1, Chunk≥1, Entity= len(memory.entities). 한 청크에 같은 엔터티가 여러 번 등장해도 Entity는 dedup.

In [ ]:
if USE_NEO4J:
    from neo4j import GraphDatabase
    drv = GraphDatabase.driver(NEO4J_URI, auth=(os.getenv('NEO4J_USER', 'neo4j'), os.getenv('NEO4J_PASSWORD')))
    with drv.session(database=client.default_database) as s:
        rows = s.run('''
          MATCH (src:Source)
          OPTIONAL MATCH (src)-[:HAS_CHUNK]->(c:Chunk)
          OPTIONAL MATCH (c)-[m:MENTIONS]->(e)
          RETURN count(DISTINCT src) AS sources,
                 count(DISTINCT c)   AS chunks,
                 count(DISTINCT e)   AS entities,
                 count(m)            AS mentions
        ''').data()
    drv.close()
    print(rows[0])
else:
    print('(ladybug embedded — neo4j 가 있을 때만 실행)')

**체크포인트**
- 같은 회사명이 청크 5개에 등장하면 Entity 노드 / MENTIONS 엣지는 각각 몇 개? (답: Entity 1, MENTIONS 5)
- Source 삭제 시 어디까지 cascade? 정책을 코드로 정한다면 어디에?

## 1.2 Extraction Prompt Design — Ontology Slice + 4-provider 비교

### 왜 ontology를 **slice** 하는가
- 전체 ontology를 prompt에 주입 → token budget 폭증, attention dilution
- intent(예: "Risk")에 관련된 class/property만 발췌 → 추출 정확도 ↑, 비용 ↓

In [ ]:
# slice_ontology — _shared.compat wrapper (PyPI vs editable both supported)
from _shared.compat import slice_ontology, slice_summary, format_ontology_block

sliced = slice_ontology(ontology, intent='company financial risk')
print('matched labels        :', sorted(sliced.matched_labels))
print('related labels        :', sorted(sliced.related_labels))
print('matched relationships :', sorted(sliced.matched_relationships)[:8])
if sliced.fallback_to_full:
    print('(no keyword match — using full ontology)')


### 4-provider Extraction A/B

같은 청크 + 같은 ontology slice → 4개 provider에 보내 **추출된 엔터티 수, schema 준수율, 토큰/지연** 비교.

In [ ]:
# Build the 3-block prompt using the slice helper from _shared.compat
ontology_block = format_ontology_block(ontology, sliced, max_classes=8, max_relationships=10)

system_b = (
    'You extract financial entities from 10-K filings.\n'
    + ontology_block + '\n\n'
    'Output strict JSON with shape: '
    '{"entities": [{"class": str, "name": str, "evidence_span": str}]}. '
    'Do not output anything outside the JSON. evidence_span MUST be a verbatim substring of the input.'
)
system_a = 'You extract financial entities from 10-K filings. Output a JSON list of entities.'

chunk = doc['text'][:1500]


In [ ]:
# Prompt A — schema 없음 (baseline)
df_a = compare_providers(
    user=chunk,
    system=system_a,
    max_tokens=600,
    response_format={'type': 'json_object'},
)
df_a

In [ ]:
# Prompt B — schema + evidence 강제
df_b = compare_providers(
    user=chunk,
    system=system_b,
    max_tokens=600,
    response_format={'type': 'json_object'},
)
df_b

In [ ]:
import pandas as pd

def parse_entities(text: str) -> int:
    """JSON 응답에서 엔터티 개수만 빠르게 카운트 (강의 시연용 — 정확도 측정 X)."""
    try:
        data = json.loads(text)
    except Exception:
        return -1
    if isinstance(data, dict) and 'entities' in data:
        return len(data['entities'])
    if isinstance(data, list):
        return len(data)
    return 0

compare = pd.DataFrame({
    'provider'      : df_a['provider'],
    'A_entities'    : df_a['response'].apply(parse_entities),
    'A_tokens'      : df_a['total_tokens'],
    'B_entities'    : df_b['response'].apply(parse_entities),
    'B_tokens'      : df_b['total_tokens'],
    'B_latency_ms'  : df_b['latency_ms'],
})
compare

**관찰 포인트** (Opik trace에서 확인)
1. schema 강제 후 엔터티 개수 분산이 좁아진다 (안정성 ↑)
2. provider별 `response_format` 처리 차이 — Kimi는 fallback path를 자동으로 탄다 (`_safe_temperature`)
3. evidence_span 강제는 hallucination을 가장 효과적으로 차단 (수치는 Ch 5 평가셋에서 정량화)

## 1.3 LPG Metadata Advantage

MENTIONS edge에 `extracted_by`, `confidence`, `extracted_at`을 직접 붙여 신뢰도 필터링이 가능.

In [ ]:
if USE_NEO4J:
    drv = GraphDatabase.driver(NEO4J_URI, auth=(os.getenv('NEO4J_USER', 'neo4j'), os.getenv('NEO4J_PASSWORD')))
    with drv.session(database=client.default_database) as s:
        # 인덱싱 시 자동으로 채워진 metadata 확인
        rows = s.run('''
          MATCH (c:Chunk)-[r:MENTIONS]->(e)
          RETURN keys(r) AS edge_props, count(*) AS n
          ORDER BY n DESC LIMIT 5
        ''').data()
    drv.close()
    for r in rows: print(r)
else:
    print('(neo4j 전용 셀)')

### 📎 깊이 있는 property 설계 — `chapter-01-property-design.md`

위 셀이 보여주는 MENTIONS edge property는 **최소 set**. 실제 production graph는 노드/엣지마다 *왜 이 property를 둬야 하는가*를 **7가지 카테고리** (Identity / Provenance / Trust / Lineage / Temporal / Tenancy / Performance) 로 분류해 설계해야 한다.

부속 문서 [`chapter-01-property-design.md`](./chapter-01-property-design.md) 에서 다룬다:
- `(:Source)` / `(:Chunk)` / `(:Entity)` 각 layer에 두는 property + *왜*
- `[:MENTIONS]` 엣지가 차단하는 실패 모드 (evidence_span, prompt_version, ontology_slice_hash, agreed_by, ...)
- `[:RELATED_TO]` 엣지의 `temporal_range`, `evidence_chunks` 다중 근거
- 카테고리별 property 매트릭스
- 강의용 미니 워크북 6문제

**원칙**: *property를 추가할 때마다 "없으면 어떤 쿼리가 깨지는지"를 적어둬라. 깨지는 쿼리가 없으면 그 property는 fluff.*

### 1.3.5 Temporal Sanity Check — "시점이 깨지면 결론이 깨진다"

Temporal property (`extracted_at`, `published_at`, `first/last_seen_at`, `temporal_range`, `version`)는 데이터 신선도와 시점 일관성의 1차 신호다. 라우팅 전에 다음 5종 검증을 자동 실행한다.

| 검증 | 무엇을 보는가 |
|---|---|
| Future-dated provenance | `extracted_at > now()` |
| Inverted temporal_range | `valid_from > valid_until` |
| Orphan extractions | `extraction_run_id` 누락 |
| Stale entities | `last_seen_at` > 1년 전 + mention_count > 0 |
| Version monotonic | 같은 source_id 의 version 이 단조 증가하는가 |

자세한 처방은 [`chapter-01-property-design.md` §10](./chapter-01-property-design.md) 참조.

In [ ]:
def temporal_sanity(database: str) -> dict:
    """Ch 1.3.5 — 5종 temporal 검증을 한 번에 실행. 모두 0이어야 깨끗."""
    if not USE_NEO4J:
        return {'skipped': 'no neo4j'}
    from neo4j import GraphDatabase
    drv = GraphDatabase.driver(NEO4J_URI, auth=(os.getenv('NEO4J_USER', 'neo4j'), os.getenv('NEO4J_PASSWORD')))
    queries = {
        'future_dated_mentions': '''
          MATCH (:Chunk)-[r:MENTIONS]->()
          WHERE r.extracted_at IS NOT NULL AND r.extracted_at > datetime()
          RETURN count(*) AS n
        ''',
        'inverted_temporal_ranges': '''
          MATCH ()-[r:RELATED_TO]->()
          WHERE r.temporal_range IS NOT NULL
            AND r.temporal_range.from IS NOT NULL
            AND r.temporal_range.to   IS NOT NULL
            AND r.temporal_range.from > r.temporal_range.to
          RETURN count(*) AS n
        ''',
        'orphan_extractions': '''
          MATCH ()-[r:MENTIONS]->()
          WHERE r.extraction_run_id IS NULL OR r.extraction_run_id = ''
          RETURN count(*) AS n
        ''',
        'stale_entities_1y': '''
          MATCH (e) WHERE NOT e:Source AND NOT e:Chunk
            AND e.last_seen_at IS NOT NULL
            AND duration.between(e.last_seen_at, datetime()).days > 365
            AND coalesce(e.mention_count, 0) > 0
          RETURN count(*) AS n
        ''',
        'non_monotonic_versions': '''
          MATCH (s:Source)
          WITH s.source_id AS sid, collect(s.version) AS vs
          WHERE size(vs) > 1
          WITH sid, vs, [i IN range(1, size(vs)-1) WHERE vs[i] <= vs[i-1]] AS bad
          WHERE size(bad) > 0
          RETURN count(*) AS n
        ''',
    }
    out = {}
    try:
        with drv.session(database=database) as s:
            for key, q in queries.items():
                try:
                    out[key] = s.run(q).single()['n']
                except Exception as exc:
                    out[key] = f'<err: {type(exc).__name__}>'
    finally:
        drv.close()
    out['clean'] = all(v == 0 for v in out.values() if isinstance(v, int))
    return out

if USE_NEO4J:
    report = temporal_sanity(client.default_database)
    for k, v in report.items():
        marker = '✅' if (isinstance(v, int) and v == 0) or (k == 'clean' and v is True) else '⚠️'
        print(f'  {marker}  {k:28s} = {v}')
else:
    print('(neo4j 필요)')

## 1.4 Community Detection (Louvain)

Entity 그래프 위에서 Louvain → community_id를 노드 property로 저장. Ch 4(라우팅)에서 "이 질문이 어느 community에 속하나"를 판단하는 근거가 된다.

In [ ]:
# 데모 효과를 위해 FinDER에서 8개 카테고리 × 2건씩 추가 인덱싱 (시간이 걸림 — LIVE 가드)
LIVE_BULK = os.getenv('CH01_BULK_INDEX', '0') == '1'
if LIVE_BULK:
    bulk = sample_per_category(2)
    for r in bulk:
        client.add(r['text'][:4000], metadata={'finder_id': r.get('id'), 'category': r.get('category')})
    print(f'indexed {len(bulk)} additional docs')
else:
    print('CH01_BULK_INDEX=1 로 환경변수 세팅 후 재실행하면 16건 추가 인덱싱 (community detection 결과 풍부해짐)')

In [ ]:
if USE_NEO4J:
    drv = GraphDatabase.driver(NEO4J_URI, auth=(os.getenv('NEO4J_USER', 'neo4j'), os.getenv('NEO4J_PASSWORD')))
    GRAPH = 'ch01-finder'
    with drv.session(database=client.default_database) as s:
        try:
            s.run(f"CALL gds.graph.drop('{GRAPH}', false)").consume()
        except Exception:
            pass
        # Entity 노드만, MENTIONS 공기(=같은 청크에 함께 등장) 그래프로 투영
        s.run(f'''
          CALL gds.graph.project.cypher(
            '{GRAPH}',
            'MATCH (e) WHERE NOT e:Source AND NOT e:Chunk RETURN id(e) AS id',
            'MATCH (e1)<-[:MENTIONS]-(c:Chunk)-[:MENTIONS]->(e2) WHERE id(e1) < id(e2) RETURN id(e1) AS source, id(e2) AS target'
          )
        ''').consume()
        louvain = s.run(f"CALL gds.louvain.write('{GRAPH}', {{writeProperty: 'community_id'}}) YIELD communityCount, modularity").data()
        dist = s.run('MATCH (e) WHERE e.community_id IS NOT NULL RETURN e.community_id AS c, count(*) AS n ORDER BY n DESC LIMIT 10').data()
    drv.close()
    print('Louvain:', louvain)
    print('community size top-10:', dist)
else:
    print('(neo4j 전용 셀)')

**체크포인트**
- community_id는 Entity property로 저장하는 게 맞을까, 별도 `(:Community)` 노드를 만들까? (답: 작은 그래프엔 property로, 큰 그래프엔 노드로 → 메타 쿼리 분리)
- 신규 문서 추가 시 community를 incremental로 갱신할까 full rebuild할까?

## 1.5 Mini Eval — 4-provider Indexing Quality

같은 5개 FinDER 문서를 4 provider 각각으로 인덱싱했을 때 추출 엔터티 수의 분산을 본다. Opik trace에서 provider별 cost/latency도 확인 가능.

In [ ]:
LIVE_EVAL = os.getenv('CH01_PROVIDER_EVAL', '0') == '1'
if not LIVE_EVAL:
    print('CH01_PROVIDER_EVAL=1 로 재실행하면 4-provider 비교 인덱싱 수행 (호출량 큼).')
else:
    eval_docs = sample_random(5, seed=7)
    eval_rows = []
    for prov in [n for n, ok in available_providers().items() if ok]:
        spec = PROVIDERS[prov]
        for d in eval_docs:
            resp = chat(
                prov,
                user=d['text'][:1500],
                system=system_b,
                temperature=0.1,
                max_tokens=600,
            )
            try:
                parsed = json.loads(resp.text)
                n_ent = len(parsed.get('entities', [])) if isinstance(parsed, dict) else 0
            except Exception:
                n_ent = -1
            eval_rows.append({
                'provider': prov, 'model': spec['model'],
                'finder_id': d.get('id'), 'category': d.get('category'),
                'entities': n_ent, 'tokens': resp.usage.get('total_tokens', 0),
            })
    eval_df = pd.DataFrame(eval_rows)
    display(eval_df)
    print()
    print('provider별 평균 entities/tokens:')
    display(eval_df.groupby('provider').agg(entities=('entities','mean'), tokens=('tokens','mean')))

## 6. Teardown

Opik 트레이스 flush. JSONL은 `./traces/chapter_01.jsonl`에 누적.

In [ ]:
teardown_opik()
print('chapter 01 done →', trace_info.get('jsonl_path'))
if trace_info.get('project'):
    print('Opik:', opik_console_link(trace_info['project']))

## 정리 & 다음 챕터

**이 챕터에서 얻은 산출물**
- 3-layer LPG에 인덱싱된 FinDER 문서들
- Entity 노드 위에 `community_id` property
- 4-provider 추출 비교 trace (Opik)

**다음 챕터 (Ch 2 — Qualification)** 에서는 위에서 만든 그래프 위에 GDS 4종 지표(Node Similarity / Degree Centrality / Clustering Coefficient / Link Prediction)를 적용하고, 각 지표를 `@function_tool`로 노출해 agent가 자율적으로 그래프 품질을 평가하게 한다.